# N-gram Text Generation from Abraham Lincoln's Speech

This notebook trains a trigram language model on the Abraham Lincoln corpus and generates new sentences that follow the style of the original speech. Because the corpus is a real historical text, the model learns formal and rhetorical patterns rather than random words.

In [353]:
from pathlib import Path
import re
import nltk
from collections import defaultdict, Counter

# ------------------------------------------------------------
# 1. Import Libraries and Define Paths
# ------------------------------------------------------------
# Path helps us locate the corpus file inside the assignment folder.
# The notebook assumes that corpus.txt already exists.

# Ensure the NLTK resources used in the lab workflow are available.
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

notebook_dir = Path.cwd()
corpus_path = notebook_dir / "corpus.txt"

# If the corpus file is missing, stop with a clear message.
if not corpus_path.exists():
    raise FileNotFoundError(f"Corpus file not found: {corpus_path}")

print("Notebook directory:", notebook_dir)
print("Corpus file:", corpus_path)
print("Corpus exists:", corpus_path.exists())

# Read the corpus from disk so the model can learn from it.
text = corpus_path.read_text(encoding="utf-8")
print("\nCorpus content preview:\n")
print(text[:1000])

Notebook directory: c:\Users\USER\OneDrive\My Codes\GitHub\NLP\Assignment_N-Gram
Corpus file: c:\Users\USER\OneDrive\My Codes\GitHub\NLP\Assignment_N-Gram\corpus.txt
Corpus exists: True

Corpus content preview:

THE PAPERS AND WRITINGS OF ABRAHAM LINCOLN

VOLUME FIVE

CONSTITUTIONAL EDITION

By Abraham Lincoln


Edited by Arthur Brooks Lapsley





THE WRITINGS OF ABRAHAM LINCOLN, Volume Five, 1858-1862




TO SYDNEY SPRING, GRAYVILLE, ILL.

SPRINGFIELD, June 19, 1858.

SYDNEY SPRING, Esq.

MY DEAR SIR:--Your letter introducing Mr. Faree was duly received. There
was no opening to nominate him for Superintendent of Public Instruction,
but through him Egypt made a most valuable contribution to the convention.
I think it may be fairly said that he came off the lion of the day--or
rather of the night. Can you not elect him to the Legislature? It seems to
me he would be hard to beat. What objection could be made to him? What is
your Senator Martin saying and doing? What is Webb about?

Plea

## 2. Preprocess the Text

The corpus is cleaned by lowercasing and removing punctuation. We also split the text into sentences so the model can learn meaningful word sequences from the Abraham Lincoln speech.

In [354]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


def full_preprocess(text, remove_stopwords=True, use_lemmatization=True):
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = word_tokenize(text)

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

    if use_lemmatization:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens


sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
processed_sentences = []
for sentence in sentences:
    sentence_tokens = full_preprocess(sentence, remove_stopwords=False, use_lemmatization=True)
    if sentence_tokens:
        processed_sentences.append(["<s>", "<s>", "<s>", *sentence_tokens, "</s>"])

processed_tokens = [token for sentence in processed_sentences for token in sentence]
print("Sentence count:", len(processed_sentences))
print("Token count:", len(processed_tokens))
print("First 30 tokens:", processed_tokens[:30])

Sentence count: 5415
Token count: 138283
First 30 tokens: ['<s>', '<s>', '<s>', 'the', 'paper', 'and', 'writing', 'of', 'abraham', 'lincoln', 'volume', 'five', 'constitutional', 'edition', 'by', 'abraham', 'lincoln', 'edited', 'by', 'arthur', 'brook', 'lapsley', 'the', 'writing', 'of', 'abraham', 'lincoln', 'volume', 'five', 'to']


## 3. Train the N-gram Model

A trigram model learns the probability of the next word from the previous two words. With a larger and more structured corpus, this produces more natural and grammatically plausible text.

In [355]:
from collections import defaultdict, Counter


def build_ngram_model(tokens, n=3):
    model = defaultdict(Counter)

    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i:i+n-1])
        next_word = tokens[i+n-1]
        model[history][next_word] += 1

    prob_model = defaultdict(dict)
    for history, counter in model.items():
        total = sum(counter.values())
        for word, count in counter.items():
            prob_model[history][word] = count / total

    return prob_model


n = 4
model = build_ngram_model(processed_tokens, n=n)

print("Sample 4-gram probabilities:")
for history, counter in list(model.items())[:10]:
    print(history, dict(counter))

Sample 4-gram probabilities:
('<s>', '<s>', '<s>') {'the': 0.0541089566020314, 'springfield': 0.008310249307479225, 'sydney': 0.00018467220683287164, 'my': 0.01809787626962142, 'faree': 0.00018467220683287164, 'there': 0.0123730378578024, 'i': 0.10156971375807941, 'can': 0.0029547553093259462, 'it': 0.040997229916897505, 'what': 0.00886426592797784, 'please': 0.004062788550323176, 'yours': 0.016435826408125576, 'lincoln': 0.04228993536472761, 'to': 0.02585410895660203, 'c': 0.004247460757156048, 'whitney': 0.00110803324099723, 'dear': 0.005170821791320407, 'give': 0.0007386888273314866, 'unless': 0.0003693444136657433, 'show': 0.00018467220683287164, 'your': 0.007940904893813482, 'w': 0.00554016620498615, 'somers': 0.0007386888273314866, 'james': 0.0007386888273314866, 'hull': 0.00018467220683287164, 'have': 0.0016620498614958448, 'he': 0.01680517082179132, 'but': 0.01809787626962142, 'that': 0.0123730378578024, 'campbell': 0.0003693444136657433, 'a': 0.030286241920590953, 'gillespie':

## 4. Generate New Sentences

This final section uses probability-based generation. The model usually picks the most likely next word, but sometimes it adds a little randomness so the output remains varied and not identical to the corpus.

In [356]:
import random


def generate_sentence(model, n=4, max_length=25, temperature=0.3):
    sentence = ["<s>", "<s>", "<s>"]

    while len(sentence) < max_length:
        history = tuple(sentence[-(n - 1):])

        if history not in model:
            break

        candidates = model[history]
        sorted_words = sorted(candidates.items(), key=lambda x: x[1], reverse=True)

        if random.random() > temperature:
            next_word = sorted_words[0][0]
        else:
            words = [w for w, _ in sorted_words]
            probs = [p for _, p in sorted_words]
            next_word = random.choices(words, weights=probs, k=1)[0]

        if next_word == "</s>":
            break

        sentence.append(next_word)

    words = sentence[3:]

    if not words:
        return ""

    sentence_text = " ".join(words)
    sentence_text = sentence_text[0].upper() + sentence_text[1:]

    if not sentence_text.endswith("."):
        sentence_text += "."

    return sentence_text


num_sentences = 100
paragraph = []
seen_sentences = set()

for _ in range(num_sentences):
    for _ in range(200):
        s = generate_sentence(model)
        if s and s not in seen_sentences and len(s.split()) >= 4:
            paragraph.append(s)
            seen_sentences.add(s)
            break

    if len(paragraph) >= num_sentences:
        break

generated_text = "\n".join(paragraph)

print("Generated Paragraph:\n")
print(generated_text)

output_path = notebook_dir / "output.txt"
output_path.write_text(generated_text + "\n", encoding="utf-8")
print(f"\nSaved generated text to {output_path}")

Generated Paragraph:

Such a result a you seem to have predicted in maine in your letter to colfax would i fear put u on.
I have not now and never have had any other than a kindly feeling toward you a to suppose that the slavery.
I have reason to hope there will be the issue.
I have not kept silence since the presidential election of silenceof avoiding public speaking of avoiding public writing.
I should think not.
I have not maintained silence from any want of real anxiety.
Dear williamyours of january th just received.
I shall be glad if you a governor of will answer by telegraph if you consent please write so at the bottom.
In view of the public peace therefore resolved that a committee of three delegate be appointed by this convention to wait upon.
I should expect it is true that exactly fifteen month ago this day i believe i have no doubt would be more.
It is much more gratifying to me that the response you give to that remark ought to justify me in closing just.
Besides this slav